# Polar plot
Based on https://github.com/climberlenny/test_openberg/blob/main/toolbox/postprocessing.py

In [16]:
cd ~/work/tutorials/sources/OpenDrift/openberg_july2026

/home/jovyan/work/tutorials/sources/OpenDrift/openberg_july2026


In [17]:
from src.utils import *
from openberg_july2026.src2.utils0 import *
from openberg_july2026.src2.utils2 import *
from openberg_july2026.src2.utils3 import *
import glob

In [18]:
# Read and subset tracker data
ib = 'iceberg2018b'
#read obs
fname = 'merged_obs10'#'merged_obs_iceberg2026e_3D'
with xr.open_dataset('./input/%s.nc'%fname) as ds:
    obs = ds.sel(iceberg=ib,trajectory_id=ib) 
    obs_seed = obs.isel(time=(obs.seed_idx==1))

In [22]:
#read sim
file_contain_all = [ib,'nextsimre_wind']
file_contain_any = ['_']
file_notcontain = ['test','idx','main','prof',]#'test','wa','next','glorys']#['topaz4','glorys','slope','era5','almost','stokes',]
file_l = glob.glob("./results/*.nc")#['./results/juliette_topaz6_topaz4_windglophynrt.nc',]
file_l = [f for f in file_l if all(ff in f for ff in file_contain_all)] #filtering for required keywords
file_l = [f for f in file_l if any(ff in f for ff in file_contain_any)] #filtering for required keywords
file_l = [f for f in file_l if all(ff not in f for ff in file_notcontain)] #filtering for neglected keywords
file_l = file_l#[::-1]
print(file_l)

['./results/iceberg2018b_gebco_sisatdtu_nextsimre_windglophyre_joined.nc', './results/iceberg2018b_gebco_nextsimre_windglophyre_joined.nc']


In [8]:
#dicts and definitions
legenddict = dict.fromkeys([f.split('/')[-1][:-3] for f in file_l])
for f in legenddict:
    ll = []
    for l in legenddict_full:
        if l.lower() in f: ll = ll+[l]
    legenddict[f]={'col':legenddict_full[ll[-1]]['col'], 'alpha':1, 'kw':', '.join(ll),'zo':legenddict_full[ll[-1]]['zo'],
                 'traj':np.arange(obs.time.size*11)}
#manual adaptions
# legenddict['iceberg2018b_gebco_topaz4_windglophyre']['zo']=25

legenddict 

{'iceberg2018b_gebco_nextsimre_windglophyre_joined': {'col': 'plum',
  'alpha': 1,
  'kw': 'WINDGLOPHYRE, ICEBERG, NEXTSIMRE',
  'zo': 40,
  'traj': array([    0,     1,     2, ..., 54942, 54943, 54944], shape=(54945,))}}

In [ ]:
try: #if not joined
    simall = xr.open_mfdataset(file_l,concat_dim="run",combine="nested")
    simall = simall.assign_coords(run=list(legenddict.keys()))
except:     #if joined already
    simall = xr.open_mfdataset(file_l) 
    #then adjust legenddict as well
    legenddict = dict.fromkeys([str(f) for f in simall.run.values])
    for f in legenddict:
        ll = []
        for l in legenddict_full:
            if l.lower() in f: ll = ll+[l]
        legenddict[f]={'col':legenddict_full[ll[-1]]['col'], 'alpha':1, 'kw':', '.join(ll),'zo':legenddict_full[ll[-1]]['zo'],
                     'traj':np.arange(obs.time.size*11)}


In [ ]:
# --- Subset---
apn = dict_analysis_period[ib][1]

# ---Trajectory subset
sim = simall
if ib in dict_traj.keys():
    for run in list(sim.run.values):
        if run in dict_traj[ib]: 
            sim = sim.sel(trajectory=dict_traj[ib][run])
            print('Trajectory subset applied on ',run)

# ---Time subset
if apn!=None: 
    sim = sim.sel(time=apn)
    print('Time subset applied: ',apn)

# ---time frequnecy subset
days = obs.time.dt.floor("D")
days2 = np.arange(obs_seed.time.dt.floor("D")[0].values,
                  obs_seed.time.dt.floor("D")[-1].values+np.timedelta64(str(obs.seed_freq.values)[:-1],str(obs.seed_freq.values)[-1]),
                  dtype='datetime64[D]')
idx = [
    np.where(days == np.datetime64(d))[0][0]
    for d in days2
    if (days == np.datetime64(d)).any()
]
obs_subtime = obs.isel(time=idx)
sim_obssubtime = sim.sel(time=obs_subtime.time, method="nearest") 

sim_sub = sim_obssubtime
obs_sub = obs_subtime

In [ ]:
# del polarplot_m
# from openberg_july2026.src2.utils3 import polarplot_m
out = polarplot_m(sim_sub,obs_sub,legenddict,title='p(%s)'%apn)

In [ ]:
#means
[{k:(float(v[0]),int(np.round(v[1])))} for k,v in out['mean'].items() if k != 'all']

In [ ]:
#spreads
[{k:(float(v[1]),int(np.round(v[-1])))} for k,v in out['spread'].items() if k != 'all']

In [ ]:
out